# 阿里 PPU JupyterLab：分钟因子 GFlowNet 训练

本 Notebook 用于在阿里 PPU/JupyterLab 计算节点运行当前分钟级 CPU 训练链路：

`本地 MemMap → NumPy 分钟算子 → Reward → GFlowNet → Alpha Pool → 因子矩阵`

训练阶段不会连接 DolphinDB，也不会重新生成 MemMap。请先把 Windows 机器生成的完整 `minute_memmap` 目录复制到 PPU 节点的本地高速磁盘，并保留 `manifest.json`、`daily_price.csv.gz`、年份目录和全部通道文件。

> 当前代码使用 PPU 节点提供的通用 CPU/PyTorch 后端，不假设 CUDA。若平台预装定制版 PyTorch，Notebook 默认不会覆盖它。

## 1. 用户参数

只需要重点修改 `MEMMAP_DIR` 和 `BLOCK_CACHE_DIR`。缓存目录应放在节点本地盘，不要放 OSS 挂载或网络共享目录。

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime

REPO_URL = 'https://gitee.com/chi-yangchen/alpha-mining-gflow-net-alpha-eval.git'
BRANCH = 'cpu-training'
SYNC_REPOSITORY = True
INSTALL_REQUIREMENTS = True
PRESERVE_PLATFORM_TORCH = True
ALLOW_CONCURRENT_RUN = False

# 修改为 PPU 节点上已经复制完成的 MemMap 目录。
MEMMAP_DIR = Path(os.environ.get('ALPHAMINING_MEMMAP_DIR', '/mnt/data/AlphaMining/minute_memmap'))
BLOCK_CACHE_DIR = Path(os.environ.get('ALPHAMINING_BLOCK_CACHE_DIR', '/mnt/data/AlphaMining/block_cache'))

LOGICAL_CPUS = os.cpu_count() or 8
TORCH_THREADS = min(32, max(4, LOGICAL_CPUS // 2))
REWARD_WORKERS = min(8, max(2, LOGICAL_CPUS // 4))
BLAS_THREADS = 1
REWARD_CHUNK_DAYS = 10
REWARD_BLOCKS_PER_TASK = 2
POOL_SIZE = 50
POOL_ATTEMPTS = 600

def locate_repository():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'configs/minute_training_cpu_ddb.yaml').exists():
            return candidate
    workspace = Path(os.environ.get('ALPHAMINING_WORKSPACE', current)).resolve()
    return workspace / 'alpha-mining-gflow-net-alpha-eval'

REPO_DIR = locate_repository()
print('REPO_DIR         =', REPO_DIR)
print('MEMMAP_DIR       =', MEMMAP_DIR)
print('BLOCK_CACHE_DIR  =', BLOCK_CACHE_DIR)
print('logical CPUs     =', LOGICAL_CPUS)
print('torch threads    =', TORCH_THREADS)
print('reward workers   =', REWARD_WORKERS)

## 2. 获取或更新仓库

如果 Notebook 已经从仓库中打开，会更新当前仓库；否则会从 Gitee 克隆。存在未提交改动时，Git 会安全停止，不会覆盖文件。

In [ ]:
if not (REPO_DIR / '.git').exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
        check=True,
    )
elif SYNC_REPOSITORY:
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

commit = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True
).strip()
print('repository commit =', commit)

## 3. 安装依赖

默认保留平台预装的 PyTorch，只安装其他依赖。训练由独立子进程启动，因此无需重启 Jupyter 内核。MemMap 模式不需要安装 DolphinDB 客户端。

In [ ]:
if INSTALL_REQUIREMENTS:
    requirement_source = REPO_DIR / 'requirements.txt'
    requirement_lines = requirement_source.read_text(encoding='utf-8').splitlines()
    if PRESERVE_PLATFORM_TORCH:
        requirement_lines = [
            line for line in requirement_lines
            if not line.strip().lower().startswith('torch')
        ]
    runtime_requirements = REPO_DIR / 'results/runtime_configs/requirements_ppu.txt'
    runtime_requirements.parent.mkdir(parents=True, exist_ok=True)
    runtime_requirements.write_text('\n'.join(requirement_lines) + '\n', encoding='utf-8')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-r', str(runtime_requirements)],
        check=True,
    )
print('dependencies ready')

## 4. 检查 PPU 节点硬件与磁盘

重点关注 CPU 核数、可用内存以及 MemMap 所在磁盘的剩余空间。

In [ ]:
import platform

print('Python       :', sys.version.replace('\n', ' '))
print('Platform     :', platform.platform())
print('Processor    :', platform.processor() or '由 lscpu 输出确认')
print('Logical CPUs :', LOGICAL_CPUS)

for command in (['lscpu'], ['free', '-h']):
    if shutil.which(command[0]):
        print('\n$', ' '.join(command))
        subprocess.run(command, check=False)

disk_target = MEMMAP_DIR if MEMMAP_DIR.exists() else MEMMAP_DIR.parent
while not disk_target.exists() and disk_target != disk_target.parent:
    disk_target = disk_target.parent
usage = shutil.disk_usage(disk_target)
print(f'\nDisk target  : {disk_target}')
print(f'Disk total   : {usage.total / 1024**3:.1f} GiB')
print(f'Disk free    : {usage.free / 1024**3:.1f} GiB')

torch_check = subprocess.run(
    [sys.executable, '-c', (
        'import torch; '
        'print(\"torch_version=\", torch.__version__); '
        'print(\"cuda_available=\", torch.cuda.is_available()); '
        'print(\"torch_threads=\", torch.get_num_threads())'
    )],
    text=True, capture_output=True, check=True,
)
print(torch_check.stdout)

## 5. 审计本地 MemMap

该步骤只读取元数据，不扫描全部分钟数组。任何检查失败都应先修复数据目录，不要直接开始训练。

In [ ]:
manifest_path = MEMMAP_DIR / 'manifest.json'
if not manifest_path.exists():
    raise FileNotFoundError(
        f'未找到 {manifest_path}。请修改 MEMMAP_DIR，或先把完整 MemMap 复制到 PPU 本地盘。'
    )
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
required_files = [
    MEMMAP_DIR / manifest.get('stocks_file', 'stocks.npy'),
    MEMMAP_DIR / manifest.get('minute_grid_file', 'minute_grid.npy'),
    MEMMAP_DIR / manifest.get('daily_file', 'daily_price.csv.gz'),
]
missing_files = [str(path) for path in required_files if not path.exists()]
if manifest.get('complete') is not True:
    raise ValueError('manifest.complete 不是 true，MemMap 构建尚未完成')
if manifest.get('n_minutes') != 241:
    raise ValueError(f"分钟网格应为241根，实际为 {manifest.get('n_minutes')}")
if missing_files:
    raise FileNotFoundError('MemMap 缺少文件：' + ', '.join(missing_files))
for year, metadata in manifest.get('years', {}).items():
    dates_file = MEMMAP_DIR / metadata['dates_file']
    if not dates_file.exists():
        raise FileNotFoundError(f'{year} 年缺少日期文件：{dates_file}')

BLOCK_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print('MemMap audit passed')
print('fingerprint =', manifest.get('fingerprint'))
print('years       =', sorted(manifest.get('years', {})))
print('dates       =', sum(item.get('n_days', 0) for item in manifest.get('years', {}).values()))
print('stocks      =', manifest.get('n_stocks'))
print('minutes/day =', manifest.get('n_minutes'))
print('channels     =', len(manifest.get('channels', [])))

## 6. 生成 PPU 运行配置

从仓库基线配置复制一份运行时配置，只修改硬件并发和输出路径。训练期仍为 2020–2023，因子池生成范围仍由原配置控制。运行配置保存在 `results/runtime_configs`，不会修改仓库基线。

In [ ]:
import yaml

base_config_path = REPO_DIR / 'configs/minute_training_cpu_ddb.yaml'
config = yaml.safe_load(base_config_path.read_text(encoding='utf-8'))
config['cpu_runtime'].update({
    'torch_threads': TORCH_THREADS,
    'interop_threads': min(4, max(1, TORCH_THREADS // 4)),
    'blas_threads': BLAS_THREADS,
})
config['dataset']['memmap'].update({
    'workers': REWARD_WORKERS,
    'reward_chunk_days': REWARD_CHUNK_DAYS,
    'reward_blocks_per_task': REWARD_BLOCKS_PER_TASK,
    'reward_parallel_backend': 'loky',
})
config['pipeline']['pool_size'] = POOL_SIZE
config['pipeline']['pool_attempts'] = POOL_ATTEMPTS
config['outputs'].update({
    'log_dir': 'results/minute_ppu/logs',
    'checkpoint': 'checkpoints/gflownet_minute_ppu_best.pt',
    'metrics': 'results/minute_ppu/gflownet_training_metrics.csv',
    'trajectory_metrics': 'results/minute_ppu/gflownet_trajectory_metrics.csv',
    'alpha_pool': 'results/minute_ppu/alpha_pool.csv',
    'factor_matrix': 'results/minute_ppu/alpha_factor_matrix.csv.gz',
})

runtime_config_dir = REPO_DIR / 'results/runtime_configs'
runtime_config_dir.mkdir(parents=True, exist_ok=True)
run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
RUNTIME_CONFIG = runtime_config_dir / f'minute_training_ppu_{run_id}.yaml'
RUNTIME_CONFIG.write_text(
    yaml.safe_dump(config, allow_unicode=True, sort_keys=False),
    encoding='utf-8',
)
print('runtime config =', RUNTIME_CONFIG)
print('training dates =', config['dataset']['mining_start_date'], '..', config['dataset']['mining_end_date'])
print('workers        =', config['dataset']['memmap']['workers'])
print('pool size      =', config['pipeline']['pool_size'])

## 7. 启动后台训练

训练使用独立进程并自动写日志。Jupyter 浏览器断开不会影响进程；但计算实例被释放、关机或管理员终止进程时训练仍会停止。再次启动会复用 `partials_v2` 和完整 Block 缓存。

In [ ]:
training_env = os.environ.copy()
training_env['ALPHAMINING_MEMMAP_DIR'] = str(MEMMAP_DIR.resolve())
training_env['ALPHAMINING_BLOCK_CACHE_DIR'] = str(BLOCK_CACHE_DIR.resolve())
training_env['PYTHONUNBUFFERED'] = '1'

TRAIN_LOG = REPO_DIR / f'results/minute_ppu/logs/minute_ppu_{run_id}.log'
TRAIN_LOG.parent.mkdir(parents=True, exist_ok=True)
PID_FILE = REPO_DIR / 'results/minute_ppu/training.pid'
STATE_FILE = REPO_DIR / 'results/minute_ppu/training_state.json'
if STATE_FILE.exists() and not ALLOW_CONCURRENT_RUN:
    previous_state = json.loads(STATE_FILE.read_text(encoding='utf-8'))
    previous_pid = int(previous_state['pid'])
    previous_log = Path(previous_state['log_file'])
    previous_text = previous_log.read_text(encoding='utf-8', errors='replace') if previous_log.exists() else ''
    previous_terminal = 'log_end status=' in previous_text
    if previous_terminal:
        previous_alive = False
    else:
        try:
            os.kill(previous_pid, 0)
        except (ProcessLookupError, PermissionError):
            previous_alive = False
        else:
            previous_alive = True
    if previous_alive:
        raise RuntimeError(
            f'检测到仍在运行的训练 PID={previous_pid}。如确认需要并行运行，请设置 ALLOW_CONCURRENT_RUN=True。'
        )
TRAIN_COMMAND = [
    sys.executable, str(REPO_DIR / 'scripts/train_cpu.py'),
    '--mode', 'minute',
    '--config', str(RUNTIME_CONFIG),
    '--threads', str(TORCH_THREADS),
    '--pool-size', str(POOL_SIZE),
    '--log-file', str(TRAIN_LOG),
]

with open(os.devnull, 'w', encoding='utf-8') as sink:
    TRAIN_PROCESS = subprocess.Popen(
        TRAIN_COMMAND,
        cwd=REPO_DIR,
        env=training_env,
        stdout=sink,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
PID_FILE.write_text(str(TRAIN_PROCESS.pid), encoding='utf-8')
run_state = {
    'pid': TRAIN_PROCESS.pid,
    'log_file': str(TRAIN_LOG.resolve()),
    'runtime_config': str(RUNTIME_CONFIG.resolve()),
    'started_at': datetime.now().astimezone().isoformat(),
    'command': TRAIN_COMMAND,
}
temporary_state = STATE_FILE.with_suffix('.json.tmp')
temporary_state.write_text(json.dumps(run_state, ensure_ascii=False, indent=2), encoding='utf-8')
temporary_state.replace(STATE_FILE)
print('training started')
print('PID      =', TRAIN_PROCESS.pid)
print('log file =', TRAIN_LOG)
print('command  =', ' '.join(TRAIN_COMMAND))

## 8. 查看状态和最新日志

可反复执行本单元格。`return_code=None` 表示当前 Notebook 启动的进程仍在运行。若内核重启，只需重新执行第1–6节，再执行本节；它会从 `training_state.json` 恢复 PID 和日志路径，不要重复执行第7节。

In [ ]:
def tail_text(path, lines=80):
    path = Path(path)
    if not path.exists():
        return f'日志尚未创建：{path}'
    with path.open('r', encoding='utf-8', errors='replace') as handle:
        content = handle.readlines()
    return ''.join(content[-lines:])

STATE_FILE = REPO_DIR / 'results/minute_ppu/training_state.json'
if not STATE_FILE.exists():
    raise FileNotFoundError('尚无 training_state.json，请先执行第7节启动训练')
run_state = json.loads(STATE_FILE.read_text(encoding='utf-8'))
pid = int(run_state['pid'])
TRAIN_LOG = Path(run_state['log_file'])
current_text = TRAIN_LOG.read_text(encoding='utf-8', errors='replace') if TRAIN_LOG.exists() else ''
if 'log_end status=completed' in current_text:
    return_code = 0
elif 'log_end status=failed' in current_text or 'log_end status=interrupted' in current_text:
    return_code = 'failed_or_interrupted'
elif 'TRAIN_PROCESS' in globals() and TRAIN_PROCESS.pid == pid:
    return_code = TRAIN_PROCESS.poll()
else:
    try:
        os.kill(pid, 0)
    except (ProcessLookupError, PermissionError):
        return_code = 'not_running'
    else:
        return_code = None
print('PID         =', pid)
print('return_code =', return_code)
print('log size MB =', round(TRAIN_LOG.stat().st_size / 1024**2, 2) if TRAIN_LOG.exists() else 0)
print('\n===== latest log =====')
print(tail_text(TRAIN_LOG, 80))

## 9. 可选：持续跟踪日志

该单元格会持续运行。需要停止显示时点击 Jupyter 的中断按钮；这只会停止 `tail`，不会停止后台训练。

In [ ]:
# 取消下一行注释后执行；使用 Jupyter 的中断按钮结束日志跟踪。
# subprocess.run(['tail', '-n', '100', '-f', str(TRAIN_LOG)], check=False)

## 10. 训练结束后检查产物

成功日志应包含 `[CPUTraining] log_end status=completed`。随后检查 checkpoint、训练指标、Alpha Pool 和因子矩阵。

In [ ]:
expected_outputs = {
    'checkpoint': REPO_DIR / config['outputs']['checkpoint'],
    'epoch metrics': REPO_DIR / config['outputs']['metrics'],
    'trajectory metrics': REPO_DIR / config['outputs']['trajectory_metrics'],
    'alpha pool': REPO_DIR / config['outputs']['alpha_pool'],
    'factor matrix': REPO_DIR / config['outputs']['factor_matrix'],
    'training log': TRAIN_LOG,
}
for name, path in expected_outputs.items():
    exists = path.exists()
    size_mb = path.stat().st_size / 1024**2 if exists else 0.0
    print(f'{name:20s} exists={str(exists):5s} size_mb={size_mb:10.2f} path={path}')

if TRAIN_LOG.exists():
    log_text = TRAIN_LOG.read_text(encoding='utf-8', errors='replace')
    if 'log_end status=completed' in log_text:
        print('\n训练状态：完成')
    elif 'log_end status=failed' in log_text:
        print('\n训练状态：失败，请查看日志末尾异常')
    else:
        print('\n训练状态：运行中、被外部终止或尚未写入结束标记')

## 11. 调参顺序

1. 先保持 `REWARD_CHUNK_DAYS=10`、`REWARD_BLOCKS_PER_TASK=2`。
2. 内存不足或 Worker 被终止：先将 `REWARD_BLOCKS_PER_TASK` 降为 1，再减少 `REWARD_WORKERS`。
3. CPU、内存和磁盘都不高：逐步增加 `REWARD_WORKERS`，每次只增加 1–2。
4. 本地高速盘且内存充足：可将 `REWARD_CHUNK_DAYS` 提高到 20，减少任务调度。
5. 观察 `[GFlowNet] stage_summary` 的 `bottleneck`，以及 `worker_read_sum_seconds` 与 `worker_compute_sum_seconds`，不要只看 CPU 利用率。
6. 不要删除 `partials_v2` 或完整 Block 缓存；异常重启依靠它们继续。